In [7]:
import os
import zipfile
import pandas as pd

# ---------- 1. EXTRACT & LOAD ----------
zip_path = "/content/store-sales-time-series-forecasting (1).zip"
extract_folder = "/content/store-sales-data/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

train    = pd.read_csv(os.path.join(extract_folder, 'train.csv'), parse_dates=['date'])
holidays = pd.read_csv(os.path.join(extract_folder, 'holidays_events.csv'), parse_dates=['date'])
stores   = pd.read_csv(os.path.join(extract_folder, 'stores.csv'))
oil      = pd.read_csv(os.path.join(extract_folder, 'oil.csv'), parse_dates=['date'])

In [12]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [13]:
train.shape

(3000888, 6)

In [14]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [15]:
holidays.shape

(350, 6)

In [16]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [17]:
stores.shape

(54, 5)

In [18]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [20]:
oil.shape

(1218, 2)

In [21]:
# ---------- 2. CLEAN ----------
# Remove store-days where the whole store sold nothing (closed)
store_day_total = train.groupby(["store_nbr", "date"])["sales"].transform("sum")
clean = train[store_day_total > 0]

# ---------- 3. FILTER TO MODELING TABLE ----------
# Grocery families that sell almost daily, plus PRODUCE
fams = ['BREAD/BAKERY', 'BEVERAGES', 'MEATS', 'CLEANING', 'DAIRY', 'DELI', 'EGGS',
        'GROCERY I', 'PERSONAL CARE', 'POULTRY', 'PREPARED FOODS', 'FROZEN FOODS',
        'GROCERY II', 'PRODUCE']

# Start 2015-06-01 (PRODUCE records complete, promotions tracked)
# Drop store 52 (only 118 days of history)
data = clean[(clean["date"] >= "2015-06-01")
             & (clean["family"].isin(fams))
             & (clean["store_nbr"] != 52)].copy()

# ---------- 4. QUICK CHECKS ----------
print(data.shape)                                # expect (589596, 6)
print(data["date"].min(), data["date"].max())    # 2015-06-01 to 2017-08-15
print(data["store_nbr"].nunique(), "stores |", data["family"].nunique(), "families")
print("Missing values:", data.isna().sum().sum())

# ---------- 1. STORES: add type, cluster, city, state ----------
df = data.merge(stores.rename(columns={"type": "store_type"}), on="store_nbr", how="left")

# ---------- 2. OIL: fill weekends/holidays, then merge by date ----------
oil_daily = (oil.set_index("date")["dcoilwtico"]
                .asfreq("D").ffill().bfill()      # fill gaps with the last known price
                .rename("oil_price").reset_index())
df = df.merge(oil_daily, on="date", how="left")

# ---------- 3. HOLIDAYS: keep real holidays, then match by locale ----------
# Drop transferred dates (not actually observed) and "Work Day" (make-up working days)
h = holidays[(~holidays["transferred"]) & (holidays["type"] != "Work Day")]

national = (h[h["locale"] == "National"][["date"]]
              .drop_duplicates().assign(national_holiday=1))
regional = (h[h["locale"] == "Regional"][["date", "locale_name"]]
              .drop_duplicates().rename(columns={"locale_name": "state"})
              .assign(regional_holiday=1))
local    = (h[h["locale"] == "Local"][["date", "locale_name"]]
              .drop_duplicates().rename(columns={"locale_name": "city"})
              .assign(local_holiday=1))

df = (df.merge(national, on="date", how="left")
        .merge(regional, on=["date", "state"], how="left")
        .merge(local,    on=["date", "city"],  how="left"))

flags = ["national_holiday", "regional_holiday", "local_holiday"]
df[flags] = df[flags].fillna(0).astype(int)

# ---------- 4. DATE FEATURES ----------
df["dow"]   = df["date"].dt.dayofweek    # 0 = Monday
df["month"] = df["date"].dt.month
df["dom"]   = df["date"].dt.day

# ---------- 5. SORT AND CHECK ----------
df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

print(df.shape)                                       # rows should still be 589,596
assert len(df) == len(data), "A merge duplicated or dropped rows"
print(df.isna().sum()[df.isna().sum() > 0])           # should print nothing
print(df[flags].mean().round(3))                      # share of rows flagged as holidays
print(df.head())

(589596, 6)
2015-06-01 00:00:00 2017-08-15 00:00:00
53 stores | 14 families
Missing values: 0
(589596, 17)
Series([], dtype: int64)
national_holiday    0.087
regional_holiday    0.000
local_holiday       0.004
dtype: float64
        id       date  store_nbr     family   sales  onpromotion   city  \
0  1566381 2015-06-01          1  BEVERAGES  2210.0            0  Quito   
1  1568163 2015-06-02          1  BEVERAGES  2439.0            1  Quito   
2  1569945 2015-06-03          1  BEVERAGES  2835.0            2  Quito   
3  1571727 2015-06-04          1  BEVERAGES  1875.0            0  Quito   
4  1573509 2015-06-05          1  BEVERAGES  2432.0            2  Quito   

       state store_type  cluster  oil_price  national_holiday  \
0  Pichincha          D       13      60.24                 0   
1  Pichincha          D       13      61.30                 0   
2  Pichincha          D       13      59.67                 0   
3  Pichincha          D       13      58.00                 0   

In [22]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,oil_price,national_holiday,regional_holiday,local_holiday,dow,month,dom
0,1566381,2015-06-01,1,BEVERAGES,2210.0,0,Quito,Pichincha,D,13,60.24,0,0,0,0,6,1
1,1568163,2015-06-02,1,BEVERAGES,2439.0,1,Quito,Pichincha,D,13,61.30,0,0,0,1,6,2
2,1569945,2015-06-03,1,BEVERAGES,2835.0,2,Quito,Pichincha,D,13,59.67,0,0,0,2,6,3
3,1571727,2015-06-04,1,BEVERAGES,1875.0,0,Quito,Pichincha,D,13,58.00,0,0,0,3,6,4
4,1573509,2015-06-05,1,BEVERAGES,2432.0,2,Quito,Pichincha,D,13,59.11,0,0,0,4,6,5


In [23]:
df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city',
       'state', 'store_type', 'cluster', 'oil_price', 'national_holiday',
       'regional_holiday', 'local_holiday', 'dow', 'month', 'dom'],
      dtype='object')

# Baseline Model (LightGBM)

In [24]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

# ---------- 1. PREPARE FEATURES & TARGET ----------
base_df = df.copy()

cat_cols = ["store_nbr", "family", "city", "state", "store_type", "cluster"]
for col in cat_cols:
    base_df[col] = base_df[col].astype("category")

feature_cols = [
    "store_nbr", "family", "onpromotion",
    "city", "state", "store_type", "cluster",
    "oil_price", "national_holiday", "regional_holiday", "local_holiday",
    "dow", "month", "dom"
]

# ---------- 2. TIME-BASED TRAIN / VALIDATION SPLIT (FINAL 16 DAYS) ----------
val_start = "2017-07-31"
train_mask = base_df["date"] < val_start
val_mask   = base_df["date"] >= val_start

X_train = base_df.loc[train_mask, feature_cols]
y_train = np.log1p(base_df.loc[train_mask, "sales"])

X_val = base_df.loc[val_mask, feature_cols]
y_val = np.log1p(base_df.loc[val_mask, "sales"])

print(f"Train rows: {len(X_train):,} | Validation rows: {len(X_val):,}")

# ---------- 3. TRAIN BASELINE LIGHTGBM ----------
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
)

# ---------- 4. EVALUATE RMSLE & FEATURE IMPORTANCE ----------
val_preds_log = model.predict(X_val)
rmsle = root_mean_squared_error(y_val, np.maximum(0, val_preds_log))
print(f"\nBaseline Validation RMSLE: {rmsle:.5f}")

# Check which of the 14 features drove the baseline
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importances:\n", imp)

Train rows: 577,724 | Validation rows: 11,872
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.096943 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 694
[LightGBM] [Info] Number of data points in the train set: 577724, number of used features: 14
[LightGBM] [Info] Start training from score 5.841703
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[801]	valid_0's l2: 0.091058

Baseline Validation RMSLE: 0.30176

Feature Importances:
 store_nbr           9653
family              5793
oil_price           2276
dow                 1818
month               1625
dom                 1461
onpromotion          999
national_holiday     295
cluster               37
local_holiday         28
city                  18
store_type            15
state                 12
regional_holiday       0
dtype: int32


The lag-free LightGBM baseline establishes a reproducible benchmark of 0.30176 RMSLE using only static store metadata, calendar features, promotions, holidays, and macro indicators. To reduce reliance on oil_price as a date proxy and capture store-level demand momentum, the next iteration will introduce 16-day shifted sales lags, rolling demand statistics, and explicit Ecuador payday indicators (15th and month-end).

# Model 2 (Refined Features)

In [26]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

# ---------- 1. DROP REDUNDANT / NOISE FEATURES ----------
# Drop deterministic store metadata (subsumed by store_nbr), raw oil_price (date proxy),
# and consolidate sparse holiday columns into a single flag.
ts_df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

ts_df["any_holiday"] = (
    (ts_df["national_holiday"] == 1) |
    (ts_df["regional_holiday"] == 1) |
    (ts_df["local_holiday"] == 1)
).astype(int)

cols_to_drop = [
    "city", "state", "store_type", "cluster",
    "oil_price", "national_holiday", "regional_holiday", "local_holiday"
]
ts_df = ts_df.drop(columns=cols_to_drop)

# ---------- 2. ADD HIGH-SIGNAL TIME-SERIES FEATURES ----------
grp = ts_df.groupby(["store_nbr", "family"])

# A. Autoregressive Sales Lags (shift >= 16 to avoid leakage over the 16-day forecast horizon)
for lag in [16, 17, 21, 28, 35]:
    ts_df[f"sales_lag_{lag}"] = grp["sales"].shift(lag)

# B. Rolling Sales Momentum (computed on sales shifted by 16 days)
shifted_sales = grp["sales"].shift(16)
shifted_grp = shifted_sales.groupby([ts_df["store_nbr"], ts_df["family"]])

ts_df["sales_roll_mean_7"]  = shifted_grp.transform(lambda s: s.rolling(7).mean())
ts_df["sales_roll_mean_28"] = shifted_grp.transform(lambda s: s.rolling(28).mean())
ts_df["sales_roll_std_7"]   = shifted_grp.transform(lambda s: s.rolling(7).std())

# C. Promotion Leads, Lags & Rolling Sum (valid because future promotions are known)
ts_df["promo_lag_1"]  = grp["onpromotion"].shift(1).fillna(0)
ts_df["promo_lead_1"] = grp["onpromotion"].shift(-1).fillna(0)
ts_df["promo_roll_7"] = grp["onpromotion"].transform(lambda s: s.rolling(7, min_periods=1).sum())

# D. Ecuador Payday & Weekend Indicators
ts_df["is_weekend"]   = (ts_df["dow"] >= 5).astype(int)
ts_df["is_payday_15"] = (ts_df["dom"] == 15).astype(int)
ts_df["is_month_end"] = ts_df["date"].dt.is_month_end.astype(int)
ts_df["is_payday"]    = ((ts_df["dom"] == 15) | ts_df["date"].dt.is_month_end).astype(int)

# ---------- 3. PREPARE MODELING MATRIX ----------
for col in ["store_nbr", "family"]:
    ts_df[col] = ts_df[col].astype("category")

# Drop the first 35 days of each series where lag_35 / roll_28 are NaN
model_df = ts_df.dropna(subset=["sales_lag_35", "sales_roll_mean_28"]).copy()

feature_cols = [
    "store_nbr", "family", "onpromotion",
    "any_holiday", "dow", "month", "dom",
    "sales_lag_16", "sales_lag_17", "sales_lag_21", "sales_lag_28", "sales_lag_35",
    "sales_roll_mean_7", "sales_roll_mean_28", "sales_roll_std_7",
    "promo_lag_1", "promo_lead_1", "promo_roll_7",
    "is_weekend", "is_payday_15", "is_month_end", "is_payday"
]

# ---------- 4. TIME-BASED TRAIN / VALIDATION SPLIT (FINAL 16 DAYS) ----------
val_start = "2017-07-31"
train_mask = model_df["date"] < val_start
val_mask   = model_df["date"] >= val_start

X_train = model_df.loc[train_mask, feature_cols]
y_train = np.log1p(model_df.loc[train_mask, "sales"])

X_val = model_df.loc[val_mask, feature_cols]
y_val = np.log1p(model_df.loc[val_mask, "sales"])

print(f"Train rows: {len(X_train):,} | Validation rows: {len(X_val):,} | Features: {len(feature_cols)}")

# ---------- 5. TRAIN TIME-SERIES LIGHTGBM & COMPARE METRICS ----------
ts_model = lgb.LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.05,
    num_leaves=63,
    subsample_for_bin=50000,
    random_state=42,
    n_jobs=-1
)

ts_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
)

val_preds_log = np.maximum(0, ts_model.predict(X_val))
ts_rmsle = root_mean_squared_error(y_val, val_preds_log)

baseline_rmsle = 0.30176
improvement = (baseline_rmsle - ts_rmsle) / baseline_rmsle * 100

print(f"\nBaseline Validation RMSLE:    {baseline_rmsle:.5f}")
print(f"Time-Series Validation RMSLE: {ts_rmsle:.5f} ({improvement:+.2f}% change)")

imp = pd.Series(ts_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 15 Feature Importances:\n", imp.head(15))

Train rows: 545,818 | Validation rows: 11,872 | Features: 22
[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056801 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 545818, number of used features: 22
[LightGBM] [Info] Start training from score 5.841782
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1149]	valid_0's l2: 0.0650232

Baseline Validation RMSLE:    0.30176
Time-Series Validation RMSLE: 0.25500 (+15.50% change)

Top 15 Feature Importances:
 store_nbr             13756
month                  6792
dom                    6721
sales_lag_28           4288
sales_lag_35           4233
family                 4

Replacing redundant static store metadata and spot oil prices with 22 leakage-free, time-series features significantly improved the LightGBM demand forecasting model, reducing validation RMSLE from 0.30176 to 0.25500 (a 15.50% accuracy improvement) and cutting log-space error variance (L2) by 28.59% (from 0.09106 to 0.06502) across the identical 16-day out-of-time validation holdout (11,872 rows spanning 2017-07-31 to 2017-08-15). In practical terms, this narrowed the typical multiplicative prediction error band from -26.0% to +35.2% down to -22.5% to +29.0% around actual daily unit sales. By pruning deterministic columns (city, state, store_type, cluster), sparse regional holiday indicators, and oil_price—which previously acted as a spurious date proxy—and introducing 16-day shifted sales lags (sales_lag_16 through sales_lag_35), rolling demand statistics (sales_roll_mean_7, sales_roll_mean_28, sales_roll_std_7), rolling promotion windows (promo_roll_7), and Ecuador semi-monthly payday flags (15th and month-end), the model shifted from memorizing macro noise to directly capturing store-family sales momentum, same-weekday shopping rhythms, and promotional lift.